In [ ]:
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
from keras.datasets import mnist

In [ ]:
(Xtr, Ltr), (X_test, L_test)=mnist.load_data() # training / test 

In [ ]:
Xtr.shape

In [ ]:
Image=Xtr[0,:,:]
Label=Ltr[0]


plt.title('Label is {Label}'.format(Label=Label))
plt.imshow(Image)

plt.show()
plt.close()


In [ ]:
fig, ax=plt.subplots(nrows=1, ncols=2, figsize=(15,15))
ax0=plt.subplot(2,2,1)
ax1=plt.subplot(2,2,2)

Image=Xtr[0,:,:]
Label=Ltr[0]
Image1=Xtr[1,:,:]
Label1=Ltr[1]


ax0.set_title('Label is {Label}'.format(Label=Label))
ax0.imshow(Image)
ax1.set_title('Label is {Label}'.format(Label=Label1))
ax1.imshow(Image1)


plt.show()
plt.close()


In [ ]:
28*28

In [ ]:
#Traing phase
num_sample=500
Tr_set=Xtr[:num_sample,:,:]
Ltr_set=Ltr[:num_sample]

Tr_set=Tr_set.reshape(num_sample,Tr_set.shape[1]*Tr_set.shape[2])

#Tr_set=Tr_set.reshape(num_sample,Tr_set.shape[1]*Tr_set.shape[2]).astype()
Tr_set.shape

Tr_set = Tr_set.astype(np.float32) # convert to float32 because of uint8 causing wraparound

In [ ]:
def predict(X):
    num_test=X.shape[0]
    Lpred=np.zeros(num_test, dtype=Ltr_set.dtype) # fill with zeroes
    #print(X[1,:])
    for i in range(num_test):
        distances=np.sum(np.abs(Tr_set-X[i,:]),axis=1)
        
        min_index= np.argmin(distances)
        Lpred[i]=Ltr_set[min_index]
    return Lpred

def predictL2(X):
    num_test=X.shape[0]
    Lpred=np.zeros(num_test, dtype=Ltr_set.dtype) # fill with zeroes
    #print(X[1,:])
    for i in range(num_test):
        distances=np.sqrt(np.sum((Tr_set-X[i,:])**2,axis=1))
        
        min_index= np.argmin(distances)
        Lpred[i]=Ltr_set[min_index]
    return Lpred

In [ ]:
Test_images=X_test.reshape(X_test.shape[0],X_test.shape[1]* X_test.shape[2])
Test_images = Test_images.astype(np.float32) # convert to float32 because of uint8 causing wraparound

Labels_predicted=predict(Test_images)

print("Accuracy L1:", np.mean(Labels_predicted==L_test))
Labels_predicted=predictL2(Test_images)
print("Accuracy L2:", np.mean(Labels_predicted==L_test))

In [ ]:

def predictKNN(X, k):
    num_test = X.shape[0]
    Lpred = np.zeros(num_test, dtype=Ltr_set.dtype)

    for i in range(num_test):
        distances = np.sqrt(
            np.sum((Tr_set - X[i, :])**2, axis=1)
        )
        nearest_indices = np.argsort(distances)[:k] # k-nearest neighbors
        nearest_labels = Ltr_set[nearest_indices]

        votes = np.bincount(nearest_labels, minlength=10) # count votes for each digit (0-9)

        Lpred[i] = np.argmax(votes) # choose digit with most votes

    return Lpred


In [ ]:
for k in list(range(1, 11)):
    predictions = predictKNN(Test_images, k)

    accuracy = np.mean(predictions == L_test)

    print(f"k = {k}, Accuracy = {accuracy:.4f}")

In [ ]:

def predictKNN_cv(X_train, y_train, X_test, k):
    num_test = X_test.shape[0]
    Lpred = np.zeros(num_test, dtype=y_train.dtype)

    for i in range(num_test):
        distances = np.sqrt(
            np.sum((X_train - X_test[i, :])**2, axis=1)
        )
        nearest_indices = np.argsort(distances)[:k]
        nearest_labels = y_train[nearest_indices]
        votes = np.bincount(nearest_labels, minlength=10)
        Lpred[i] = np.argmax(votes)

    return Lpred


In [ ]:
# 3-fold

k_values = list(range(1, 11))

rng = np.random.default_rng(67)
indices = rng.permutation(len(Tr_set))

# divide into 3 folds
folds = np.array_split(indices, 3)

results = {}

for k in k_values:
    accuracies = []

    for i in range(3):
        
        val_idx = folds[i] # validation fold

        train_idx = np.concatenate([ # rest becomes training fold
            folds[j] for j in range(3) if j != i
        ])

        X_train = Tr_set[train_idx]
        y_train = Ltr_set[train_idx]

        X_val = Tr_set[val_idx]
        y_val = Ltr_set[val_idx]

        predictions = predictKNN_cv(
            X_train, y_train, X_val, k
        )

        accuracy = np.mean(predictions == y_val)
        accuracies.append(accuracy)

    results[k] = np.mean(accuracies)

    print(f"k={k}: {accuracies}, mean={results[k]:.4f}")

best_k = max(results, key=results.get)

print("Best k:", best_k)


In [ ]:
predictions = predictKNN_cv(
    Tr_set,
    Ltr_set,
    Test_images,
    best_k
)

test_accuracy = np.mean(predictions == L_test)

print("Best k:", best_k)
print(f"Test accuracy: {test_accuracy:.4f}")